# Incident reports — generate a run & explore

This notebook can **trigger a full ingestion run** and then **explore** its
output interactively.

Golden rule: it **reuses** the logic from `src/` (runner, loader, config) and
**never duplicates** it. The exact same orchestration powers
`scripts/run_ingestion.py`. Once an analysis becomes stable and useful, promote
it into `src/` and expose it through the CLI.

## 0. Setup

Make the `src` package importable and enable inline plots. `config` paths are
absolute (derived from `src/config.py`), so they work regardless of the kernel's
working directory.

In [ ]:
import sys
from pathlib import Path

# Resolve the project root whether the kernel starts in notebooks/ or the repo root.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

from src import config

print("Project root:", PROJECT_ROOT)
print("Artifacts dir:", config.ARTIFACTS_DIR)

## 1. Generate a fresh run (or pick the latest)

`run_from_env` reads `ANONYMIZATION_SALT` from `.env`, runs the whole pipeline
(load → anonymise → enrich) and writes all artifacts into a new timestamped
folder, exactly like the CLI.

- Set `GENERATE_NEW_RUN = True` to produce a brand-new run.
- Set it to `False` to only explore the most recent existing run.

> Each generation creates a **new** timestamped folder; it never overwrites a
> previous run. Anonymisation is deterministic, so the data is identical across
> runs (same salt).

In [ ]:
from src.ingestion.runner import run_from_env

GENERATE_NEW_RUN = True  # set to False to only explore the latest existing run

if GENERATE_NEW_RUN:
    run_dir = run_from_env(config.DEFAULT_INPUT_CSV)
else:
    runs = sorted(
        p for p in config.ARTIFACTS_DIR.iterdir() if p.is_dir() and p.name.isdigit()
    )
    assert runs, "No run found. Set GENERATE_NEW_RUN = True to create one."
    run_dir = runs[-1]

RUN_ID = run_dir.name
print("Active run:", RUN_ID)
for f in sorted(run_dir.iterdir()):
    print("  -", f.name)

## 2. Load the anonymised dataset

No salt is required here: the data produced by the run is already anonymised.

In [ ]:
df = pd.read_csv(run_dir / "incidents_anonymized.csv", parse_dates=[config.DATE_COLUMN])
df.head()

## 3. Dataset overview

Shape, dtypes and missing values at a glance.

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## 4. Reuse production code from `src/`

Instead of recomputing metrics by hand, call the same function used by the
pipeline. This guarantees consistency with the runs registry.

In [ ]:
from src.ingestion.loader import compute_quality_metrics

compute_quality_metrics(df)

## 5. Custom exploration (beyond the standard plots)

### 5.1 Average reporting confidence per machine

Which machines tend to be flagged by a single isolated signal (low confidence)
versus several concurrent signals (high confidence)?

In [ ]:
by_machine = (
    df.groupby(config.MACHINE_COLUMN)[config.CONFIDENCE_COLUMN]
    .mean()
    .sort_values()
)

ax = by_machine.plot.barh(figsize=(8, 5), color="#4C72B0")
ax.set_xlabel("Mean confidence index")
ax.set_title("Average reporting confidence per machine")
plt.tight_layout()
plt.show()

### 5.2 Severity by shift

Cross-tabulate severity against the work shift to spot risky time slots.

In [ ]:
severity_num = pd.to_numeric(df[config.SEVERITY_COLUMN], errors="coerce")
crosstab = df.assign(severity=severity_num).pivot_table(
    index=config.SHIFT_COLUMN,
    values="severity",
    aggfunc=["count", "mean"],
)
crosstab

## 6. Promoting an analysis to production

When a cell above proves useful and stable:

1. Move the logic into the right module under `src/` (e.g. a new function in
   `src/visualization/`).
2. Wire it into `src/ingestion/runner.py` so every run produces it.
3. Add a unit test if it carries business logic.

Keep this notebook for exploration only — it is versioned in Git but **not**
tracked by DVC (see `CLAUDE.md`).